# rng-throughput
This notebook measures the throughput of random number generation as implemented by the C++ standard library.

```python
for dist in distributions:
    for gen in generators:
        tic()
        for i in range(n_iters):
            x = dist(gen)

        time_s = toc()
        throughput = sizeof(x) * n_iters / time_s
```

The benchmark can be implemented with the python language and benefits from:
  * string interpolation
  * JIT compilation
  * LLVM passes to detect *dead code elimination*

In [1]:
# usual llvm initialization
import llvmlite
import llvmlite.binding as llvm

from peppo.opt import usual_optimize
from peppo.ext_source import ExtSource, Language

llvm.initialize_native_target()
llvm.initialize_native_asmprinter()

## Domain specific code
Users of the PEPPO framework declare the program logic using primitive data and functions.
The user code is ingested with the `ExtSource` class, which is responsible of compiling the code into LLVM IR.

Once an LLVM module is created, the limited functionality (as exposed by the `llvmlite` dependency) can be accessed.
This involves optimization passes and JIT compilation, but in principle custom passes can be added.

In [2]:
distributions = [
    'normal_distribution<double>',
    'uniform_real_distribution<double>'
]

generators = [
    'minstd_rand0',
    'minstd_rand',
    'mt19937',
    'mt19937_64',
    'ranlux24_base',
    'ranlux48_base',
    'ranlux24',
    'ranlux48',
    'knuth_b',
]

In [3]:
def generate_benchmark(entry_point: str,
                     distribution: str,
                     generator: str) -> ExtSource:
    # digraphs `<%` are used to not escape the curly braces
    src = f"""
    #include <chrono>
    #include <random>

    using namespace std;

    double {entry_point}(std::size_t nruns) <%
        {distribution} dist;
        {generator} gen;

        const auto t_start = std::chrono::steady_clock::now();
        for (; nruns; --nruns) <%
            const auto x = dist(gen);
        %>
        const auto t_end = std::chrono::steady_clock::now();

        const std::chrono::duration<double> delta = t_end - t_start;
    %>
    """

    return ExtSource(src, Language.CPP)

## Exploration with the PEPPO framework

In [7]:
target = llvm.Target.from_default_triple()
target_machine = target.create_target_machine()

In [4]:
entry_point = 'profile_generation'
bench = generate_benchmark(entry_point, 'normal_distribution<double>', 'mt19937')
print(bench.src)


    #include <chrono>
    #include <random>

    using namespace std;

    double profile_generation(std::size_t nruns) <%
        normal_distribution<double> dist;
        mt19937 gen;

        const auto t_start = std::chrono::steady_clock::now();
        for (; nruns; --nruns) <%
            const auto x = dist(gen);
        %>
        const auto t_end = std::chrono::steady_clock::now();

        const std::chrono::duration<double> delta = t_end - t_start;
    %>
    


In [5]:
llvm_module = bench.compile_to_llvm_ir()
print(llvm_module)

; ModuleID = '<string>'
source_filename = "-"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-i128:128-f80:128-n8:16:32:64-S128"
target triple = "x86_64-pc-linux-gnu"

%"class.std::normal_distribution" = type <{ %"struct.std::normal_distribution<>::param_type", double, i8, [7 x i8] }>
%"struct.std::normal_distribution<>::param_type" = type { double, double }
%"class.std::mersenne_twister_engine" = type { [624 x i64], i64 }
%"class.std::chrono::time_point" = type { %"class.std::chrono::duration" }
%"class.std::chrono::duration" = type { i64 }
%"class.std::chrono::duration.0" = type { double }
%"struct.std::__detail::_Adaptor" = type { ptr }

$_ZNSt19normal_distributionIdEC2Ev = comdat any

$_ZNSt23mersenne_twister_engineImLm32ELm624ELm397ELm31ELm2567483615ELm11ELm4294967295ELm7ELm2636928640ELm15ELm4022730752ELm18ELm1812433253EEC2Ev = comdat any

$_ZNSt19normal_distributionIdEclISt23mersenne_twister_engineImLm32ELm624ELm397ELm31ELm2567483615ELm11ELm4294967295ELm7ELm263

In [8]:
original_module = llvm_module.clone()
usual_optimize(
    target_machine,
    llvm_module,
)

print(llvm_module)

; ModuleID = '<string>'
source_filename = "-"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-i128:128-f80:128-n8:16:32:64-S128"
target triple = "x86_64-pc-linux-gnu"

%"class.std::normal_distribution" = type <{ %"struct.std::normal_distribution<>::param_type", double, i8, [7 x i8] }>
%"struct.std::normal_distribution<>::param_type" = type { double, double }
%"class.std::mersenne_twister_engine" = type { [624 x i64], i64 }

$_ZNSt19normal_distributionIdEclISt23mersenne_twister_engineImLm32ELm624ELm397ELm31ELm2567483615ELm11ELm4294967295ELm7ELm2636928640ELm15ELm4022730752ELm18ELm1812433253EEEEdRT_RKNS0_10param_typeE = comdat any

$_ZNSt23mersenne_twister_engineImLm32ELm624ELm397ELm31ELm2567483615ELm11ELm4294967295ELm7ELm2636928640ELm15ELm4022730752ELm18ELm1812433253EEclEv = comdat any

; Function Attrs: mustprogress noreturn uwtable
define dso_local noundef double @_Z18profile_generationm(i64 noundef %0) local_unnamed_addr #0 {
  %2 = alloca %"class.std::normal_distri